[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/solutions/34_gpt2_block_solution.ipynb)

# 🔴 Solution: GPT-2 Transformer Block

*Attention & Transformers · Hard*

Reference implementation. Try it yourself in `34_gpt2_block.ipynb` first.

---
Implement one **GPT-2 transformer block** as an `nnx.Module`: pre-norm residual
attention followed by pre-norm residual MLP.

$$h = x + \mathrm{Attn}(\mathrm{LN}_1(x)), \qquad y = h + \mathrm{MLP}(\mathrm{LN}_2(h))$$

with causal multi-head self-attention

$$\mathrm{Attn}(x) = \mathrm{Concat}_h\!\left[
\mathrm{softmax}\!\left(\frac{Q_h K_h^\top}{\sqrt{d_h}} + M\right) V_h
\right] W_O,
\qquad M_{ij} = \begin{cases} 0 & j \le i \\ -\infty & j > i \end{cases}$$

(the softmax runs per head, over the key axis; the heads are concatenated back
to width $D$ before the output projection)

and a position-wise MLP $\;d \to 4d \to \mathrm{GELU} \to d$.

### Rules
- Subclass `nnx.Module`; signature `GPT2Block(d_model, num_heads, *, rngs)`
- `__call__(x)` maps `(B, T, D) -> (B, T, D)`, deterministic (no dropout)
- Expose the four sub-parts as attributes: `self.ln1`, `self.attn`, `self.ln2`,
  `self.mlp` — each callable `(B, T, D) -> (B, T, D)`
- `nnx.Linear` and `nnx.LayerNorm` are allowed building blocks;
  `nnx.MultiHeadAttention` and `nnx.dot_product_attention` are **banned**
- Attention must be genuinely multi-head: split `D` into `num_heads` heads of
  size `d_head = D // num_heads` and scale scores by $1/\sqrt{d_h}$
- Attention must be **causal**: position $i$ may attend to $j \le i$ only
- MLP hidden width is exactly `4 * d_model`, activation GELU

### Pre-norm vs post-norm — the whole point of this problem
The 2017 "Attention Is All You Need" block was **post-norm**:

$$x \leftarrow \mathrm{LN}(x + \mathrm{Attn}(x))$$

GPT-2 moved the norm inside the branch, and every large model since has kept it
there. That one move is most of why 48-layer GPT-2 XL, 96-layer GPT-3 and
80-layer Llama-3-70B stacks train at all.

Write the network as a chain of blocks and look at the backward path. Pre-norm
gives $y = x + f(\mathrm{LN}(x))$, so $\partial y/\partial x = I + \partial
f/\partial x$: there is an **untouched additive highway** from the embedding to
the logits, and the gradient at layer 0 is the sum of a clean identity term plus
each block's contribution. Post-norm puts a LayerNorm *on* that highway. LN's
Jacobian rescales by $1/\sigma$ and projects out the mean direction, so the
signal is multiplied by $L$ such factors on the way down. At depth the product
drifts — which is exactly why the original Transformer needed learning-rate
**warmup** and careful init, and why pre-norm nets *can* be trained without it
(Xiong et al., 2020). Do not overclaim in an interview: warmup is still standard
practice for pre-norm models, it just stops being load-bearing.

The price: nothing bounds the residual stream any more. Each block adds an
$O(1)$ correction, so $\mathrm{Var}(x_\ell)$ grows roughly linearly in depth,
and the last block's output is *not* normalized. That is why GPT-2 has a final
`ln_f` before the unembedding — a detail people forget when they hand-roll the
stack, and a good follow-up question.

### Traps hiding in the shapes
- Mask **before** the softmax, on the logits, not on the probabilities.
  Zeroing probabilities afterwards leaves the rows un-normalized.
- Reshape to `(B, T, H, d_h)` then `swapaxes(1, 2)`. Reshaping straight to
  `(B, H, T, d_h)` silently interleaves the heads with the time axis and still
  produces the right output *shape* — a bug tests based on shape alone never catch.
- Divide by $\sqrt{d_h}$, the **per-head** dimension, not by $\sqrt{D}$.

In [ ]:
# Install jax-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q jax-judge flax')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp
from flax import nnx

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✅ REFERENCE SOLUTION

import jax
import jax.numpy as jnp
from flax import nnx


class CausalSelfAttention(nnx.Module):
    def __init__(self, d_model: int, num_heads: int, *, rngs: nnx.Rngs):
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.num_heads = num_heads
        self.d_head = d_model // num_heads
        # One fused projection for q, k, v — same params as three separate ones.
        self.qkv = nnx.Linear(d_model, 3 * d_model, rngs=rngs)
        self.out = nnx.Linear(d_model, d_model, rngs=rngs)

    def _split_heads(self, t, B, T):
        # (B, T, D) -> (B, T, H, Dh) -> (B, H, T, Dh): heads become a batch axis.
        return t.reshape(B, T, self.num_heads, self.d_head).swapaxes(1, 2)

    def __call__(self, x):
        B, T, D = x.shape
        q, k, v = jnp.split(self.qkv(x), 3, axis=-1)
        q = self._split_heads(q, B, T)
        k = self._split_heads(k, B, T)
        v = self._split_heads(v, B, T)

        # (B, H, T, T); scale by the PER-HEAD dim.
        scores = (q @ jnp.swapaxes(k, -1, -2)) / jnp.sqrt(jnp.float32(self.d_head))

        causal = jnp.tril(jnp.ones((T, T), dtype=bool))
        scores = jnp.where(causal, scores, -jnp.inf)   # mask the LOGITS
        weights = jax.nn.softmax(scores, axis=-1)

        y = weights @ v                                # (B, H, T, Dh)
        y = y.swapaxes(1, 2).reshape(B, T, D)          # merge heads back
        return self.out(y)


class MLP(nnx.Module):
    def __init__(self, d_model: int, *, rngs: nnx.Rngs):
        self.fc = nnx.Linear(d_model, 4 * d_model, rngs=rngs)
        self.proj = nnx.Linear(4 * d_model, d_model, rngs=rngs)

    def __call__(self, x):
        return self.proj(jax.nn.gelu(self.fc(x)))


class GPT2Block(nnx.Module):
    def __init__(self, d_model: int, num_heads: int, *, rngs: nnx.Rngs):
        self.d_model = d_model
        self.num_heads = num_heads
        self.ln1 = nnx.LayerNorm(d_model, rngs=rngs)
        self.attn = CausalSelfAttention(d_model, num_heads, rngs=rngs)
        self.ln2 = nnx.LayerNorm(d_model, rngs=rngs)
        self.mlp = MLP(d_model, rngs=rngs)

    def __call__(self, x):
        # Pre-norm: the norm sits INSIDE each branch, never on the residual stream.
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

In [ ]:
# 🔍 Verify
import jax
import jax.numpy as jnp
from flax import nnx

block = GPT2Block(d_model=64, num_heads=4, rngs=nnx.Rngs(0))
x = jax.random.normal(jax.random.key(0), (2, 8, 64))
print("out:", block(x).shape)

# The residual highway: a big input passes through almost untouched, because each
# branch sees a NORMALISED copy and writes back only an O(1) correction.
big = x * 50.0
delta = block(big) - big
print(f"input std {float(jnp.std(big)):.2f} -> block wrote a delta of std {float(jnp.std(delta)):.2f}")

# Parameter budget: 12*d^2 + 13*d for a bias-everywhere GPT-2 block.
total = sum(int(p.size) for p in jax.tree.leaves(nnx.state(block, nnx.Param)))
print("params:", total, "vs 12*d^2 + 13*d =", 12 * 64 ** 2 + 13 * 64)

# Causality: rewriting the future leaves the past bit-identical.
y1 = block(x)
y2 = block(x.at[:, 4:].set(0.0))
print("max change in positions 0..3:", float(jnp.abs(y1[:, :4] - y2[:, :4]).max()))

In [ ]:
# Run the judge against the reference solution
from jax_judge import check

check("gpt2_block")